In [ ]:
import yaml
from pathlib import Path

_REPO_ROOT = Path("../..") 
with open(_REPO_ROOT / "configs/data_paths.yaml") as _f:
    _paths = yaml.safe_load(_f)
SAR_ROOT = _paths["SAR_sea_ice_dataset"]


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
from datetime import datetime
import rasterio
from rasterio.plot import plotting_extent

In [ ]:
import matplotlib as mpl

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

def fig_textwidth(height_ratio=0.62):
    import matplotlib.pyplot as plt
    return plt.subplots(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio))

In [ ]:
setup_pub_style(fontsize=9)

In [ ]:
triplets_path = f'{SAR_ROOT}/triplets/region-27_0-82_951-35_52-83_8_triplets_dt24h_24h_tol1h.csv'
# above domain row 0 is the dirverginf leads case.

triplets_path = f'{SAR_ROOT}/triplets/region-27_2-82_348-35_0-83_2_triplets_dt24h_24h_tol1h.csv'

In [ ]:
def to_db(x, eps=1e-6):
    return 10 * np.log10(np.maximum(x.astype(np.float32), eps))


def fname_to_dt(path):
    base = os.path.splitext(os.path.basename(path))[0]
    try:
        dt = datetime.strptime(base, "%Y%m%dT%H%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except:
        return base


def plot_triplet(csv_path, row=0, channels=(0,),
                 keys=("t_minus_24", "t", "t_plus_24"),
                 labels=(r"$t-\Delta t$", r"$t$", r"$t+\Delta t$"),
                 cmap="gray"):

    df = pd.read_csv(csv_path)

    panels = []
    all_vals = []

    for key, lab in zip(keys, labels):
        path = str(df.loc[row, key])
        with rasterio.open(path) as src:
            extent = plotting_extent(src)
            aspect = abs(src.transform.a) / abs(src.transform.e)

            for ch in channels:
                arr = to_db(src.read(ch + 1))
                panels.append((arr, extent, aspect, lab, ch, path))
                all_vals.append(arr[np.isfinite(arr)])

    all_vals = np.concatenate([v.ravel() for v in all_vals])
    vmin, vmax = np.nanpercentile(all_vals, [1, 99])

    n = len(panels)
    fig = plt.figure(figsize=(5 * n + 0.6, 5))
    gs = gridspec.GridSpec(1, n + 1, width_ratios=[1] * n + [0.05], wspace=0.12)

    axes = [fig.add_subplot(gs[0, i]) for i in range(n)]
    cax = fig.add_subplot(gs[0, -1])

    im = None

    for i, (ax, (arr, extent, aspect, lab, ch, path)) in enumerate(zip(axes, panels)):
        left, right, bottom, top = extent

        im = ax.imshow(arr, extent=extent, origin="upper",
                       cmap=cmap, vmin=vmin, vmax=vmax,
                       interpolation="nearest")
        ax.set_aspect(aspect)

        pol = "HH" if ch == 0 else "HV"
        dt_str = fname_to_dt(path)

        ax.set_title(f"{lab} ({pol}-Pol)\n{dt_str}", fontsize=11)

        ax.set_xlabel("Lon")

        ax.grid(color='white')

        ax.set_xticks(np.linspace(left, right, 5))
        ax.set_yticks(np.linspace(top, bottom, 5))
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        if i == 0:
            ax.set_ylabel("Lat")
        else:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Backscatter (dB)")

    return fig, axes


# Example:
fig, axes = plot_triplet(triplets_path, row=309, channels=(0,))
plt.show()



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
from datetime import datetime
import rasterio
from rasterio.plot import plotting_extent
import seaborn as sns

# assumes you already ran: setup_pub_style(fontsize=9)
# and have TEXTWIDTH_IN available from your style module


def to_db(x, eps=1e-6):
    return 10 * np.log10(np.maximum(x.astype(np.float32), eps))


def fname_to_dt(path):
    base = os.path.splitext(os.path.basename(path))[0]
    try:
        dt = datetime.strptime(base, "%Y%m%dT%H%M")
        return dt.strftime("%Y-%m-%d %H:%M")
    except Exception:
        return base


def plot_triplet(
    csv_path,
    row=0,
    channels=(0,),
    keys=("t_minus_24", "t", "t_plus_24"),
    labels=(r"$t-\Delta t$", r"$t$", r"$t+\Delta t$"),
    cmap="gray",
    height_ratio=0.55,
):
    df = pd.read_csv(csv_path)

    panels, all_vals = [], []

    for key, lab in zip(keys, labels):
        path = str(df.loc[row, key])
        with rasterio.open(path) as src:
            extent = plotting_extent(src)
            aspect = abs(src.transform.a) / abs(src.transform.e)

            for ch in channels:
                arr = to_db(src.read(ch + 1))
                panels.append((arr, extent, aspect, lab, ch, path))
                all_vals.append(arr[np.isfinite(arr)])

    all_vals = np.concatenate([v.ravel() for v in all_vals])
    vmin, vmax = np.nanpercentile(all_vals, [1, 99])

    n = len(panels)

    # --- LaTeX-sized figure ---
    W = TEXTWIDTH_IN
    H = TEXTWIDTH_IN * height_ratio
    fig = plt.figure(figsize=(W, H))

    gs = gridspec.GridSpec(1, n + 1, width_ratios=[1] * n + [0.05], wspace=0.0001)
    axes = [fig.add_subplot(gs[0, i]) for i in range(n)]
    cax = fig.add_subplot(gs[0, -1])

    im = None
    for i, (ax, (arr, extent, aspect, lab, ch, path)) in enumerate(zip(axes, panels)):
        left, right, bottom, top = extent

        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
        )
        ax.set_aspect(aspect)

        pol = "HH" if ch == 0 else "HV"
        dt_str = fname_to_dt(path)
        ax.set_title(f"{lab} ({pol}-Pol)\n{dt_str}")  # fontsize handled by rcParams

        palette = sns.color_palette("colorblind")
        red_color = palette[7]   # index 3 is the red-ish one

        # ax.set_xlabel("Lon")
        ax.grid(color='white', lw=.6)

        # Keep grid, but hide ticks and labels
        ax.set_xticks(np.linspace(left, right, 5))
        ax.set_yticks(np.linspace(top, bottom, 5))

        # Hide tick marks and labels
        ax.tick_params(
            axis="both",
            which="both",
            length=0,          # remove tick marks
            labelbottom=False,
            labelleft=False
        )

        # Remove axis labels entirely
        ax.set_xlabel("")
        ax.set_ylabel("")

        # ax.set_xticks(np.linspace(left, right, 5))
        # ax.set_yticks(np.linspace(top, bottom, 5))
        # ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        # ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

        # if i == 3:
        #     ax.set_ylabel("Lat")
        #     ax.tick_params(axis="y", labelrotation=45)
        # else:
        #     ax.set_ylabel("")
        #     ax.tick_params(labelleft=False)
        
        # if i == 3:
        #     ax.set_xlabel("Lon")
        # else:
        #     ax.set_xlabel("")

        # # ax.tick_params(axis="x", labelrotation=45)
        # ax.tick_params(labelbottom=False)


    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Backscatter (dB)")

    fig.tight_layout()
    return fig, axes


In [ ]:
setup_pub_style(fontsize=8)

fig, axes = plot_triplet(triplets_path, row=791, channels=(1,), height_ratio=0.3)
fig.savefig("triplet.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.patheffects as pe
from scipy.ndimage import distance_transform_edt

def plot_sar_triplet_channels(
    csv_path,
    row=0,
    key="t",
    inc_band=3,
    height_ratio=0.3,
    hh_lims=None,
    hv_lims=None,
):
    df = pd.read_csv(csv_path)
    path = str(df.loc[row, key])
    dt_str = fname_to_dt(path)
    print(dt_str)

    with rasterio.open(path) as src:
        extent = plotting_extent(src)
        aspect = abs(src.transform.a) / abs(src.transform.e)
        hh  = to_db(src.read(1))
        hv  = to_db(src.read(2))
        mask = hv <= -59.9
        if mask.any():
            _, idx = distance_transform_edt(mask, return_indices=True)
            hv[mask] = hv[tuple(idx[:, mask])]
        inc = src.read(inc_band).astype(np.float32)

    panels = [
        (hh,  "HH Backscatter [dB]",  "gray_r",  hh_lims),
        (hv,  "HV Backscatter [dB]",  "gray_r",  hv_lims),
        (inc, "Incidence Angle [°]",   "gray_r",  None),
    ]

    W = TEXTWIDTH_IN
    H = TEXTWIDTH_IN * height_ratio
    fig = plt.figure(figsize=(W, H))

    gs = gridspec.GridSpec(
        2, 3,
        height_ratios=[1, 0.06],
        hspace=0.0,
        wspace=0.05,
    )

    left, right, bottom, top = extent

    def _annotate(ax, text, xy, xytext, color, outline):
        stroke = [pe.withStroke(linewidth=1.5, foreground=outline)]
        ann = ax.annotate(
            text,
            xy=xy,
            xytext=xytext,
            fontsize=8,
            color=color,
            arrowprops=dict(arrowstyle="->", color=color, lw=0.8),
        )
        ann.set_path_effects(stroke)
        ann.arrow_patch.set_path_effects(stroke)
        return ann

    def _arrow_only(ax, xy, xytext, color, outline):
        stroke = [pe.withStroke(linewidth=1.5, foreground=outline)]
        ann = ax.annotate(
            "",
            xy=xy,
            xytext=xytext,
            arrowprops=dict(arrowstyle="->", color=color, lw=0.8),
        )
        ann.arrow_patch.set_path_effects(stroke)

    for i, (arr, label, cmap, lims) in enumerate(panels):
        ax  = fig.add_subplot(gs[0, i])
        cax = fig.add_subplot(gs[1, i])

        finite = arr[np.isfinite(arr)]
        vmin, vmax = lims if lims is not None else np.nanpercentile(finite, [1, 99])

        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
        )
        ax.set_aspect(aspect)

        if i < 2:
            color   = "black" if i == 0 else "white"
            outline = "white" if i == 0 else "black"

            # Floe annotation
            _annotate(
                ax, "Floe",
                xy=((left + right) / 2 - 1.1, (top + bottom) / 2 - 0.04),
                xytext=(left + 0.3, bottom + 0.5),
                color=color, outline=outline,
            )

            # Lead annotation (text + first arrow)
            _annotate(
                ax, "Lead",
                xy=(left + 5.8, (top + bottom) / 2 - 0.2),
                xytext=(left + 4.1, bottom + 0.6),
                color=color, outline=outline,
            )
            # Lead second arrow only
            _arrow_only(
                ax,
                xy=(left + 2.65, (top + bottom) / 2 + 0.25),
                xytext=(left + 4.1, bottom + 0.6),
                color=color, outline=outline,
            )

        ax.set_xticks(np.linspace(left, right, 5))
        ax.set_yticks(np.linspace(top, bottom, 5))
        ax.tick_params(axis="both", which="both", length=0,
                       labelbottom=False, labelleft=False)

        cb = fig.colorbar(im, cax=cax, orientation="horizontal")
        cb.set_label(label)
        cb.ax.tick_params(labelsize=7)
        cb.ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))

    return fig

In [ ]:
row=1346
fig = plot_sar_triplet_channels(triplets_path, row=row, height_ratio=0.39)
fig.savefig("sar_HH_HV_IA.pdf", bbox_inches="tight")
plt.show()